# 01 — Padronização, consolidação, concisão e artefatos finais

Este notebook organiza os resultados de avaliação da ferramenta e dos chats comerciais, gera batches para avaliação de concisão, incorpora `concisao_score` e `justificativa_concisao`, calcula `avaliacao_final`, mede o tamanho da resposta final com `tiktoken` e salva os artefatos finais em `eval/artefatos`.

A regra de `avaliacao_final` é:

- quando `avaliacao_gpt` e `avaliacao_opus` convergem, usa essa avaliação convergente;
- quando divergem, usa `avaliacao_humana`.

Observação sobre tokens:

- `output_tokens` representa os tokens de saída reportados pela API durante o fluxo de execução/julgamento;
- `resposta_tokens_tiktoken` representa apenas o tamanho da resposta final armazenada na coluna `resposta`, medido com `tiktoken`;
- `custo_estimado_usd` usa uma estimativa simplificada, sem prompt cache/cache hit/cache write.


## Etapa 1 — Canonicalização dos arquivos `.xlsx`

Renomeia `{empresa}__{provider}__{modelo}__{timestamp}[__corrigido].xlsx` → `{empresa}_{modelo}.xlsx`.

**Regras de segurança (idempotência):**
- Arquivos já canônicos são listados como `[JÁ CANÔNICO]` e mantidos intactos.
- Se um arquivo bruto aparecer e o destino canônico já existir, é marcado como `[CONFLITO]` e **nada é sobrescrito nem deletado** — você decide manualmente o que fazer.
- Pareamento `corrigido` vs `normal`: quando os dois existem, mantém apenas o `corrigido`.

**Contadores no resumo final:**
- `renomeado(s)`: arquivos brutos transformados em canônicos nesta execução.
- `já canônico(s)`: arquivos que já estavam no formato final antes desta execução.
- `conflito(s)`: arquivos brutos ignorados porque o destino canônico já existia.

In [1]:
import re
from pathlib import Path
from collections import defaultdict

# -----------------------------------------------------------------------------
# Configuração de paths
# -----------------------------------------------------------------------------
# Este notebook foi preparado para ficar em:
# /home/julio/Documentos/tcc_GENAI/v8/edital-assistant/eval/notebooks/
#
# Os paths abaixo usam PROJECT_ROOT/EVAL_ROOT para evitar depender do diretório
# de execução do kernel no VS Code/Jupyter.
PROJECT_ROOT = Path("/home/julio/Documentos/tcc_GENAI/v8/edital-assistant")
EVAL_ROOT = PROJECT_ROOT / "eval"

RESULTS_DIR = EVAL_ROOT / "resultados_ferramenta"

PATTERN = re.compile(
    r'^(?P<empresa>[^_]+)__'
    r'(?P<provider>[^_]+)__'
    r'(?P<modelo>.+?)__'
    r'\d{8}_\d{6}'
    r'(?P<corrigido>__corrigido)?'
    r'\.xlsx$'
)

def canonical_name(empresa, modelo):
    modelo_norm = modelo.replace("__", "-")
    return f"{empresa}_{modelo_norm}"

groups = defaultdict(lambda: {"corrigido": None, "normal": None})

# Separa arquivos em "brutos" (com timestamp, casam com PATTERN) e "já canônicos".
all_xlsx = list(RESULTS_DIR.glob("*.xlsx"))
raw_files = []
ja_canonicos = []
for f in all_xlsx:
    m = PATTERN.match(f.name)
    if m:
        raw_files.append((f, m))
    else:
        ja_canonicos.append(f)

for f, m in raw_files:
    key = canonical_name(m.group("empresa"), m.group("modelo"))
    if m.group("corrigido"):
        groups[key]["corrigido"] = f
    else:
        groups[key]["normal"] = f

renomeados, conflitos = 0, 0

# Lista os arquivos já canonicalizados (estado preservado do run anterior).
for f in sorted(ja_canonicos):
    print(f"[JÁ CANÔNICO] {f.name}")

for key, files in sorted(groups.items()):
    target = RESULTS_DIR / f"{key}.xlsx"

    # IDEMPOTÊNCIA: se já existe arquivo canônico, não sobrescreve nem deleta.
    if target.exists():
        print(f"[CONFLITO]    {target.name} já existe — arquivos brutos ignorados.")
        conflitos += 1
        continue

    if files["corrigido"] and files["normal"]:
        files["normal"].unlink()
        files["corrigido"].rename(target)
        print(f"[RENOMEADO]   {target.name}  (corrigido escolhido; normal removido)")
        renomeados += 1
    elif files["corrigido"]:
        files["corrigido"].rename(target)
        print(f"[RENOMEADO]   {target.name}  (corrigido)")
        renomeados += 1
    elif files["normal"]:
        files["normal"].rename(target)
        print(f"[RENOMEADO]   {target.name}  (normal)")
        renomeados += 1

print(
    f"\nResumo: {renomeados} renomeado(s) | "
    f"{len(ja_canonicos)} já canônico(s) | "
    f"{conflitos} conflito(s)"
)
print("Concluído!")

[JÁ CANÔNICO] bndes_claude-haiku-4-5.xlsx
[JÁ CANÔNICO] bndes_claude-opus-4-7.xlsx
[JÁ CANÔNICO] bndes_claude-sonnet-4-6.xlsx
[JÁ CANÔNICO] bndes_deepseek-v4-flash.xlsx
[JÁ CANÔNICO] bndes_deepseek-v4-pro.xlsx
[JÁ CANÔNICO] bndes_gpt-4o-mini.xlsx
[JÁ CANÔNICO] bndes_gpt-5.4-mini.xlsx
[JÁ CANÔNICO] bndes_gpt-5.4.xlsx
[JÁ CANÔNICO] bndes_gpt-5.5.xlsx
[JÁ CANÔNICO] cvm_claude-haiku-4-5.xlsx
[JÁ CANÔNICO] cvm_claude-opus-4-7.xlsx
[JÁ CANÔNICO] cvm_claude-sonnet-4-6.xlsx
[JÁ CANÔNICO] cvm_deepseek-v4-flash.xlsx
[JÁ CANÔNICO] cvm_deepseek-v4-pro.xlsx
[JÁ CANÔNICO] cvm_gpt-4o-mini.xlsx
[JÁ CANÔNICO] cvm_gpt-5.4-mini.xlsx
[JÁ CANÔNICO] cvm_gpt-5.4.xlsx
[JÁ CANÔNICO] cvm_gpt-5.5.xlsx
[JÁ CANÔNICO] petrobras_claude-haiku-4-5.xlsx
[JÁ CANÔNICO] petrobras_claude-opus-4-7.xlsx
[JÁ CANÔNICO] petrobras_claude-sonnet-4-6.xlsx
[JÁ CANÔNICO] petrobras_deepseek-v4-flash.xlsx
[JÁ CANÔNICO] petrobras_deepseek-v4-pro.xlsx
[JÁ CANÔNICO] petrobras_gpt-4o-mini.xlsx
[JÁ CANÔNICO] petrobras_gpt-5.4-mini.xlsx
[JÁ

## Etapa 2 — Consolidação, tokens e custo estimado

Junta os dois DataFrames de cada pasta (juiz GPT e juiz Opus), corrige a contagem de tokens, mede o tamanho da resposta final com `tiktoken` e adiciona `custo_estimado_usd`.

A ordem das colunas de tokens/custo fica assim:

```text
resposta | resposta_tokens_tiktoken | input_tokens | output_tokens | custo_estimado_usd
```

Assim, `custo_estimado_usd` fica depois das duas medições operacionais de tokens.

### Diferença importante

- `output_tokens`: tokens de saída reportados pela API. Em fluxos com agente/ferramentas/julgamento, pode incluir saídas intermediárias, não apenas a resposta final.
- `resposta_tokens_tiktoken`: tokens da resposta final que ficou salva em `resposta`. É uma métrica útil para concisão, mas não substitui a métrica de cobrança nativa do provedor.

### Estimativa de custo

A estimativa é propositalmente simples:

```text
(input_tokens * preço_input + output_tokens * preço_output) / 1_000_000
```

Não considera prompt cache, cache hit, cache write, Batch API, data residency, endpoint regional/multirregional, long context ou cobrança específica de ferramentas server-side.

### Claude Opus 4.7 e tokenização

Há uma configuração opcional para análise de sensibilidade de custo do `claude-opus-4-7`, caso os tokens disponíveis tenham sido estimados por um tokenizer externo e não sejam tokens nativos reportados pela API do provedor.

Por padrão, o ajuste fica **desligado** para evitar dupla contagem quando `input_tokens` e `output_tokens` já vierem da API.


In [2]:
import re
from collections import defaultdict
from pathlib import Path

import pandas as pd

try:
    import tiktoken
except ImportError as exc:
    raise ImportError(
        "Este notebook usa tiktoken para medir resposta_tokens_tiktoken. "
        "Instale com: pip install tiktoken"
    ) from exc

FOLDERS_DIR = EVAL_ROOT / "avaliacao_judge_llms_tool"

# -----------------------------------------------------------------------------
# Tokenização da resposta final
# -----------------------------------------------------------------------------
# Usamos tiktoken como métrica padronizada de tamanho da resposta final.
# Para modelos OpenAI, tenta usar encoding_for_model; para Claude/DeepSeek e
# nomes não conhecidos pelo tiktoken, usa o200k_base como proxy consistente.
_TOKEN_ENCODING_MEMO = {}


def _get_encoding_for_model(modelo=None):
    chave = modelo or "__default__"
    if chave in _TOKEN_ENCODING_MEMO:
        return _TOKEN_ENCODING_MEMO[chave]

    enc = None
    if modelo and str(modelo).startswith("gpt-"):
        try:
            enc = tiktoken.encoding_for_model(str(modelo))
        except KeyError:
            enc = None

    if enc is None:
        enc = tiktoken.get_encoding("o200k_base")

    _TOKEN_ENCODING_MEMO[chave] = enc
    return enc


def contar_tokens_tiktoken(texto, modelo=None):
    """Conta tokens do texto final salvo em `resposta` usando tiktoken."""
    if texto is None:
        return 0
    try:
        if pd.isna(texto):
            return 0
    except (TypeError, ValueError):
        pass

    texto = str(texto)
    if not texto:
        return 0
    return len(_get_encoding_for_model(modelo).encode(texto))


# -----------------------------------------------------------------------------
# Preços e estimativa simplificada
# -----------------------------------------------------------------------------
# USD por 1M tokens. A simplificação adotada aqui NÃO considera prompt cache,
# Batch API, data residency, endpoint regional/multirregional, long context ou
# cobrança específica de ferramentas server-side.
PRECOS_USD_POR_MTOK = {
    # Anthropic Claude
    "claude-haiku-4-5":  {"input": 1.00, "output": 5.00},
    "claude-sonnet-4-6": {"input": 3.00, "output": 15.00},
    "claude-opus-4-7":   {"input": 5.00, "output": 25.00},

    # DeepSeek — preço cheio para V4 Pro, sem aplicar promoção temporária.
    "deepseek-v4-flash": {"input": 0.14, "output": 0.28},
    "deepseek-v4-pro":   {"input": 1.74, "output": 3.48},

    # OpenAI
    "gpt-4o-mini":  {"input": 0.15, "output": 0.60},
    "gpt-5.4-mini": {"input": 0.75, "output": 4.50},
    "gpt-5.4":      {"input": 2.50, "output": 15.00},
    "gpt-5.5":      {"input": 5.00, "output": 30.00},
}

# Ajuste opcional APENAS para estimativa de custo, sem mexer em
# `resposta_tokens_tiktoken`, `input_tokens` ou `output_tokens`.
#
# Contexto: a documentação da Anthropic indica que o tokenizer novo do Opus 4.7
# pode usar mais tokens para o mesmo texto fixo. Isso só deve ser aplicado se os
# tokens disponíveis no arquivo forem estimativas de tokenizer externo. Se os
# tokens vieram da API do provedor, eles já refletem a tokenização nativa, então
# mantenha o ajuste desligado para evitar dupla contagem.
APLICAR_AJUSTE_TOKENIZACAO_OPUS47_NO_CUSTO = False
FATOR_TOKENIZACAO_OPUS47_CUSTO = 1.35  # use 1.30 se preferir a hipótese conservadora "~1,3x"

FATORES_TOKENS_CUSTO_POR_MODELO = {
    "claude-opus-4-7": (
        FATOR_TOKENIZACAO_OPUS47_CUSTO
        if APLICAR_AJUSTE_TOKENIZACAO_OPUS47_NO_CUSTO
        else 1.0
    ),
}


def _num_col(df, col, default=0):
    """Retorna coluna numérica; se não existir, retorna série default."""
    if col in df.columns:
        return pd.to_numeric(df[col], errors="coerce").fillna(default)
    return pd.Series(default, index=df.index, dtype="float64")


def _preparar_colunas_tokens_api(df_origem):
    """
    Prepara colunas de tokens de forma simples.

    - input_tokens = input_tokens, com fallback para prompt_tokens.
    - output_tokens = output_tokens, com fallback para completion_tokens.
    - não usa nenhuma coluna de prompt cache/cache hit/cache write.
    """
    if "input_tokens" in df_origem.columns:
        input_tokens = _num_col(df_origem, "input_tokens", 0)
    else:
        input_tokens = _num_col(df_origem, "prompt_tokens", 0)

    if "output_tokens" in df_origem.columns:
        output_tokens = _num_col(df_origem, "output_tokens", 0)
    else:
        output_tokens = _num_col(df_origem, "completion_tokens", 0)

    return pd.DataFrame({
        "input_tokens": input_tokens.round().astype("Int64"),
        "output_tokens": output_tokens.round().astype("Int64"),
    }, index=df_origem.index)


def _calcular_custo_simplificado(tokens_df, precos, modelo_str):
    """Calcula custo por linha sem considerar cache ou descontos."""
    if precos is None:
        return pd.Series([pd.NA] * len(tokens_df), index=tokens_df.index, dtype="Float64")

    input_tokens = pd.to_numeric(tokens_df["input_tokens"], errors="coerce").fillna(0)
    output_tokens = pd.to_numeric(tokens_df["output_tokens"], errors="coerce").fillna(0)
    fator = FATORES_TOKENS_CUSTO_POR_MODELO.get(str(modelo_str), 1.0)

    input_para_custo = input_tokens * fator
    output_para_custo = output_tokens * fator

    return (
        (input_para_custo / 1_000_000) * precos["input"]
      + (output_para_custo / 1_000_000) * precos["output"]
    ).astype("Float64")


def _reordenar_colunas_principais(df):
    """Mantém tokens e custo em ordem legível."""
    preferidas = [
        "edital", "modelo", "id", "categoria", "pergunta", "resposta",
        "resposta_tokens_tiktoken",
        "input_tokens", "output_tokens", "custo_estimado_usd",
        "n_invocacoes", "latencia_s",
    ]
    cols = [c for c in preferidas if c in df.columns]
    cols += [c for c in df.columns if c not in cols]
    return df[cols]


dfs = []
modelos_sem_preco = set()

for folder in sorted(FOLDERS_DIR.iterdir()):
    if not folder.is_dir():
        continue

    xlsx_files = list(folder.glob("*.xlsx"))
    if len(xlsx_files) != 2:
        print(f"[SKIP] {folder.name}: {len(xlsx_files)} arquivo(s)")
        continue

    file_a, file_b = pd.read_excel(xlsx_files[0]), pd.read_excel(xlsx_files[1])

    def get_judge_col(df, kind):
        return next((c for c in df.columns if kind in c), None)

    if get_judge_col(file_a, "gpt"):
        df_gpt, df_opus = file_a, file_b
    else:
        df_gpt, df_opus = file_b, file_a

    # Identificação edital/modelo a partir do nome da pasta.
    parts = folder.name.split("_", 1)
    edital = parts[0]
    modelo_str = parts[1] if len(parts) > 1 else folder.name

    base_cols = ["id", "categoria", "pergunta", "resposta", "n_invocacoes", "latencia_s"]
    faltantes_base = [c for c in base_cols if c not in df_gpt.columns]
    if faltantes_base:
        raise ValueError(f"{folder.name}: colunas base ausentes: {faltantes_base}")

    df = df_gpt[base_cols].copy()

    # Tokens da resposta final salva — diferente de output_tokens do fluxo completo.
    df.insert(
        df.columns.get_loc("resposta") + 1,
        "resposta_tokens_tiktoken",
        [contar_tokens_tiktoken(txt, modelo_str) for txt in df["resposta"]],
    )

    # Tokens reportados/registrados no fluxo, sem lógica de cache.
    tokens_df = _preparar_colunas_tokens_api(df_gpt)
    for col in tokens_df.columns:
        df[col] = tokens_df[col]

    # Custo estimado simplificado: input + output, sem cache/descontos.
    precos = PRECOS_USD_POR_MTOK.get(modelo_str)
    if precos is None:
        modelos_sem_preco.add(modelo_str)
    df["custo_estimado_usd"] = _calcular_custo_simplificado(tokens_df, precos, modelo_str)

    df["avaliacao_gpt"]      = df_gpt[get_judge_col(df_gpt, "avaliacao")]
    df["justificativa_gpt"]  = df_gpt[get_judge_col(df_gpt, "justificativa")]
    df["avaliacao_opus"]     = df_opus[get_judge_col(df_opus, "avaliacao")]
    df["justificativa_opus"] = df_opus[get_judge_col(df_opus, "justificativa")]
    df["convergencia"]       = df["avaliacao_gpt"] == df["avaliacao_opus"]

    df.insert(0, "edital", edital)
    df.insert(1, "modelo", modelo_str)
    df = _reordenar_colunas_principais(df)

    dfs.append(df)

if modelos_sem_preco:
    print(f"[AVISO] Modelos sem preço cadastrado (custo = NA): {sorted(modelos_sem_preco)}")

result = pd.concat(dfs, ignore_index=True)
print(result.shape)
result.head(3)


(1350, 17)


,edital,modelo,id,categoria,pergunta,resposta,resposta_tokens_tiktoken,input_tokens,output_tokens,custo_estimado_usd,n_invocacoes,latencia_s,avaliacao_gpt,justificativa_gpt,avaliacao_opus,justificativa_opus,convergencia
0,bndes,claude-haiku-4-5,1,Dados sobre as inscrições,Qual o período de inscrição para o concurso BN...,**Período de inscrição do BNDES (01/2024):**\n...,143,10951,307,0.012486,2,4.91,1.0,A resposta está alinhada ao edital: o item 6.2...,1.0,"O edital, no item 6.2.1, estabelece o período ...",True
1,bndes,claude-haiku-4-5,2,Dados sobre as inscrições,Qual o valor da taxa de inscrição do concurso ...,A **taxa de inscrição do concurso BNDES é de R...,203,10440,362,0.01225,2,5.25,1.0,Correta. O item 6.2.3 fixa o valor de inscriçã...,1.0,O item 6.2.3 fixa expressamente o valor de R$ ...,True
2,bndes,claude-haiku-4-5,3,Dados sobre as inscrições,Em qual site o candidato deve fazer a inscriçã...,O candidato deve fazer a inscrição no **site d...,113,10079,257,0.011364,2,4.38,1.0,Correta. Os itens 6.2.1 e 6.3.1 determinam que...,1.0,O item 6.2.1 e o 6.3.1 indicam expressamente q...,True


## Etapa 3 — Alinhamento com a avaliação humana

Carrega `divergentes_human_tool_eval.xlsx` e cruza com `result` usando `(pasta, id)` como chave.

**3.1 — Verificação em 4 níveis:**
1. Contagem: nº de divergentes em `result` == nº de linhas no arquivo humano?
2. Chaves: o conjunto `(pasta, id)` é exatamente o mesmo dos dois lados?
3. Conteúdo: para cada `(pasta, id)` casada, `categoria`, `pergunta` e `resposta` batem? (pega drift de dados — se alguém regerou os arquivos e o mesmo `id` agora aponta pra outra pergunta).
4. Duplicatas em `(pasta, id)` no humano (quebraria o merge silenciosamente).

**3.2 — Merge:** anexa `avaliacao_humana` e `modelo_correto` ao `result`. Linhas com `convergencia=True` ficam com `NaN` (não tinham divergência pra ser avaliada manualmente).

In [3]:
# --- Etapa 3.1: verificação de alinhamento ---

HUMAN_EVAL_PATH = EVAL_ROOT / "human_divergence_eval" / "divergentes_human_tool_eval.xlsx"

df_human = pd.read_excel(HUMAN_EVAL_PATH)
print("Colunas no arquivo humano:", list(df_human.columns))

# Chave `pasta` em result (edital + modelo) para casar com o arquivo humano
result["pasta"] = result["edital"] + "_" + result["modelo"]
divergentes = result[~result["convergencia"]].copy()

# 1) Contagem
print(f"\nDivergentes em result (convergencia=False): {len(divergentes)}")
print(f"Linhas no arquivo humano:                   {len(df_human)}")
print(f"Quantidade igual?                           {len(divergentes) == len(df_human)}")

# 2) Conjunto de chaves (pasta, id)
keys_r = set(zip(divergentes["pasta"], divergentes["id"]))
keys_h = set(zip(df_human["pasta"], df_human["id"]))
faltam_no_humano   = keys_r - keys_h
sobrando_no_humano = keys_h - keys_r
print(f"Chaves (pasta, id) idênticas?               {not faltam_no_humano and not sobrando_no_humano}")
if faltam_no_humano:
    print(f"  faltam no humano (até 10): {sorted(faltam_no_humano)[:10]}")
if sobrando_no_humano:
    print(f"  sobram no humano (até 10): {sorted(sobrando_no_humano)[:10]}")

# 3) Conteúdo: para cada (pasta, id) casada, pergunta/resposta/categoria batem?
check = divergentes.merge(
    df_human[["pasta", "id", "categoria", "pergunta", "resposta"]],
    on=["pasta", "id"],
    how="inner",
    suffixes=("_result", "_human"),
)
for col in ["categoria", "pergunta", "resposta"]:
    igual = (check[f"{col}_result"] == check[f"{col}_human"]).all()
    print(f"`{col}` igual em todas as linhas casadas?    {igual}")
    if not igual:
        diff = check[check[f"{col}_result"] != check[f"{col}_human"]]
        print(f"  {len(diff)} divergência(s). Primeiras:")
        print(diff[["pasta", "id", f"{col}_result", f"{col}_human"]].head().to_string(index=False))

# 4) Duplicatas em (pasta, id) no humano — silenciariam um bug no merge
dup_human = df_human.duplicated(subset=["pasta", "id"], keep=False)
if dup_human.any():
    print(f"\n[!] {dup_human.sum()} linha(s) com (pasta, id) duplicada no humano:")
    print(df_human.loc[dup_human, ["pasta", "id"]].sort_values(["pasta", "id"]).to_string(index=False))

Colunas no arquivo humano: ['id', 'categoria', 'pergunta', 'resposta', 'justificativa_gpt', 'avaliacao_gpt', 'justificativa_opus', 'avaliacao_opus', 'convergencia', 'pasta', 'Avaliação Humana', 'Modelo Correto']

Divergentes em result (convergencia=False): 92
Linhas no arquivo humano:                   92
Quantidade igual?                           True
Chaves (pasta, id) idênticas?               True
`categoria` igual em todas as linhas casadas?    True
`pergunta` igual em todas as linhas casadas?    True
`resposta` igual em todas as linhas casadas?    True


In [4]:
# --- Etapa 3.2: merge da avaliação humana em `result` ---

# Renomeia para snake_case (consistência com avaliacao_gpt / avaliacao_opus)
df_human_cols = df_human.rename(columns={
    "Avaliação Humana": "avaliacao_humana",
    "Modelo Correto":   "modelo_correto",
})[["pasta", "id", "avaliacao_humana", "modelo_correto"]]

n_antes = len(result)
result = result.merge(df_human_cols, on=["pasta", "id"], how="left")

# Sanity checks ANTES do fillna (após o fillna, notna() retornaria sempre len(result))
n_humanas = result["avaliacao_humana"].notna().sum()
n_div     = (~result["convergencia"]).sum()
print(f"Linhas em result: {len(result)} (antes: {n_antes})")
print(f"avaliacao_humana preenchida em: {n_humanas}")
print(f"Divergentes (convergencia=False): {n_div}")
print(f"Bate?                             {len(result) == n_antes and n_humanas == n_div}")

# Preenche convergentes (sem divergência → sem avaliação humana) com placeholder
result = result.drop(columns=["pasta"]).fillna({
    "avaliacao_humana": "nao pertinente",
    "modelo_correto":   "nao pertinente",
})

result.head()

Linhas em result: 1350 (antes: 1350)
avaliacao_humana preenchida em: 92
Divergentes (convergencia=False): 92
Bate?                             True


,edital,modelo,id,categoria,pergunta,resposta,resposta_tokens_tiktoken,input_tokens,output_tokens,custo_estimado_usd,n_invocacoes,latencia_s,avaliacao_gpt,justificativa_gpt,avaliacao_opus,justificativa_opus,convergencia,avaliacao_humana,modelo_correto
0,bndes,claude-haiku-4-5,1,Dados sobre as inscrições,Qual o período de inscrição para o concurso BN...,**Período de inscrição do BNDES (01/2024):**\n...,143,10951,307,0.012486,2,4.91,1.0,A resposta está alinhada ao edital: o item 6.2...,1.0,"O edital, no item 6.2.1, estabelece o período ...",True,nao pertinente,nao pertinente
1,bndes,claude-haiku-4-5,2,Dados sobre as inscrições,Qual o valor da taxa de inscrição do concurso ...,A **taxa de inscrição do concurso BNDES é de R...,203,10440,362,0.01225,2,5.25,1.0,Correta. O item 6.2.3 fixa o valor de inscriçã...,1.0,O item 6.2.3 fixa expressamente o valor de R$ ...,True,nao pertinente,nao pertinente
2,bndes,claude-haiku-4-5,3,Dados sobre as inscrições,Em qual site o candidato deve fazer a inscriçã...,O candidato deve fazer a inscrição no **site d...,113,10079,257,0.011364,2,4.38,1.0,Correta. Os itens 6.2.1 e 6.3.1 determinam que...,1.0,O item 6.2.1 e o 6.3.1 indicam expressamente q...,True,nao pertinente,nao pertinente
3,bndes,claude-haiku-4-5,4,Dados sobre as inscrições,Quais são as formas de pagamento da taxa de in...,"Com base no edital do BNDES, as formas de paga...",205,10304,381,0.012209,2,5.66,1.0,Correta. O item 6.3.2 prevê exatamente duas fo...,1.0,O item 6.3.2 lista exatamente as duas formas d...,True,nao pertinente,nao pertinente
4,bndes,claude-haiku-4-5,5,Dados sobre as inscrições,Qual é o prazo-limite de pagamento da inscriçã...,"Com base no edital do BNDES, o **prazo-limite ...",346,11284,568,0.014124,2,7.11,1.0,Correta. O item 6.3.2 estabelece que o pagamen...,1.0,O item 6.3.2 estabelece que o pagamento deve o...,True,nao pertinente,nao pertinente


## Etapa 4 — Consolidação dos chats comerciais

Paralela à Etapa 2, mas para os arquivos da pasta `avaliacao_judge_llms_chats_comerciais/`.

**Diferenças vs. Etapa 2:**
- Arquivos planos (não em subpastas): padrão `{empresa}_{modelo}_{judge}_eval.xlsx`, com `judge ∈ {gpt, opus}` e `modelo ∈ {std_chatgpt, std_claude}`.
- Sem métricas operacionais (`n_invocacoes`, `latencia_s`, tokens, custo) — interface web não expõe isso.
- `categoria` é puxada por lookup `(edital, id)` do `result` já consolidado (mesmo conjunto de perguntas em ambos os pipelines).

Saída: `result_chats` com colunas `edital, modelo, id, categoria, pergunta, resposta, avaliacao_gpt, justificativa_gpt, avaliacao_opus, justificativa_opus, convergencia`.

In [5]:
# --- Etapa 4: consolidação dos chats comerciais ---

CHATS_DIR = EVAL_ROOT / "avaliacao_judge_llms_chats_comerciais"

CHAT_PATTERN = re.compile(
    r'^(?P<empresa>[^_]+)_(?P<modelo>.+?)_(?P<judge>gpt|opus)_eval\.xlsx$'
)

# Agrupa arquivos por (empresa, modelo) — dois juízes por par
chat_groups = defaultdict(dict)
for f in sorted(CHATS_DIR.glob("*.xlsx")):
    m = CHAT_PATTERN.match(f.name)
    if not m:
        print(f"[SKIP] {f.name}: não bate com o padrão")
        continue
    key = (m.group("empresa"), m.group("modelo"))
    chat_groups[key][m.group("judge")] = f

# Lookup de categoria a partir de `result` — chats e ferramenta respondem o mesmo conjunto de perguntas
cat_lookup = (
    result.drop_duplicates(["edital", "id"])
          .set_index(["edital", "id"])["categoria"]
)

dfs_chats = []
for (empresa, modelo), judges in sorted(chat_groups.items()):
    if "gpt" not in judges or "opus" not in judges:
        print(f"[SKIP] {empresa}/{modelo}: faltam juízes (presentes: {list(judges)})")
        continue

    df_gpt  = pd.read_excel(judges["gpt"])
    df_opus = pd.read_excel(judges["opus"])

    # Sanity: ids alinhados linha a linha entre os dois juízes
    assert (df_gpt["id"].values == df_opus["id"].values).all(), (
        f"IDs desalinhados em {empresa}/{modelo}"
    )

    df = df_gpt[["id", "pergunta", "resposta"]].copy()
    df.insert(
        df.columns.get_loc("resposta") + 1,
        "resposta_tokens_tiktoken",
        [contar_tokens_tiktoken(txt, modelo) for txt in df["resposta"]],
    )

    df["avaliacao_gpt"]      = df_gpt["avaliacao_gpt"]
    df["justificativa_gpt"]  = df_gpt["justificativa_gpt"]
    df["avaliacao_opus"]     = df_opus["avaliacao_opus"]
    df["justificativa_opus"] = df_opus["justificativa_opus"]
    df["convergencia"]       = df["avaliacao_gpt"] == df["avaliacao_opus"]

    df.insert(0, "edital", empresa)
    df.insert(1, "modelo", modelo)
    df.insert(3, "categoria", [cat_lookup.get((empresa, i), pd.NA) for i in df["id"]])

    dfs_chats.append(df)

result_chats = pd.concat(dfs_chats, ignore_index=True)

# Aviso se alguma categoria não bateu
n_sem_cat = result_chats["categoria"].isna().sum()
if n_sem_cat:
    print(f"[AVISO] {n_sem_cat} linha(s) sem categoria (não bateram com result)")

result_chats = _reordenar_colunas_principais(result_chats)
print(result_chats.shape)
result_chats.head(3)


(300, 12)


,edital,modelo,id,categoria,pergunta,resposta,resposta_tokens_tiktoken,avaliacao_gpt,justificativa_gpt,avaliacao_opus,justificativa_opus,convergencia
0,bndes,std_chatgpt,1,Dados sobre as inscrições,Qual o período de inscrição para o concurso BN...,O período de inscrição para a Seleção Pública ...,98,1.0,Certo. O item 6.2.1 (p. 9) prevê inscrição ent...,1.0,Subitens 6.2.1 e 6.3.1 do edital: inscrições n...,True
1,bndes,std_chatgpt,2,Dados sobre as inscrições,Qual o valor da taxa de inscrição do concurso ...,O valor da taxa de inscrição do concurso do\nB...,40,1.0,Certo. O item 6.2.3 (p. 9) fixa o recolhimento...,1.0,Subitem 6.2.3 do edital: 'O recolhimento do va...,True
2,bndes,std_chatgpt,3,Dados sobre as inscrições,Em qual site o candidato deve fazer a inscriçã...,A inscrição do concurso do\nBanco Nacional de ...,46,1.0,Certo. Os itens 6.2.1 e 6.3.1 (p. 9-10) indica...,1.0,Subitem 6.2.1 do edital: inscrição na página d...,True


## Etapa 5 — Alinhamento com a avaliação humana dos chats

Carrega `chats_div_avaliado.xlsx` e cruza com `result_chats` usando `(pasta, id)` como chave. Mesmas 4 verificações da Etapa 3.

**Pegadinhas do arquivo humano dos chats** (diferenças vs. `divergentes_human_tool_eval.xlsx`):
- Coluna humana se chama `avaliação humana` (minúscula, com acento) — no arquivo da ferramenta era `Avaliação Humana` (title case).
- A coluna "modelo correto" foi exportada como `modelo.1` (colisão pandas com a outra coluna `modelo` do export). Renomeada para `modelo_correto`.
- Tem uma coluna `Unnamed: 0` (índice antigo do export) que descartamos no `read_excel`.

In [6]:
# --- Etapa 5.1: verificação de alinhamento ---

HUMAN_CHATS_PATH = EVAL_ROOT / "human_divergence_eval" / "chats_div_avaliado.xlsx"

df_human_chats = pd.read_excel(HUMAN_CHATS_PATH).drop(columns=["Unnamed: 0"], errors="ignore")
print("Colunas no arquivo humano (chats):", list(df_human_chats.columns))

# Constrói chave `pasta` dos dois lados
result_chats["pasta"] = result_chats["edital"] + "_" + result_chats["modelo"]
df_human_chats["pasta"] = df_human_chats["edital"] + "_" + df_human_chats["modelo"]

divergentes_chats = result_chats[~result_chats["convergencia"]].copy()

# 1) Contagem
print(f"\nDivergentes em result_chats (convergencia=False): {len(divergentes_chats)}")
print(f"Linhas no arquivo humano:                          {len(df_human_chats)}")
print(f"Quantidade igual?                                  {len(divergentes_chats) == len(df_human_chats)}")

# 2) Conjunto de chaves (pasta, id)
keys_r = set(zip(divergentes_chats["pasta"], divergentes_chats["id"]))
keys_h = set(zip(df_human_chats["pasta"], df_human_chats["id"]))
faltam_no_humano   = keys_r - keys_h
sobrando_no_humano = keys_h - keys_r
print(f"Chaves (pasta, id) idênticas?                      {not faltam_no_humano and not sobrando_no_humano}")
if faltam_no_humano:
    print(f"  faltam no humano (até 10): {sorted(faltam_no_humano)[:10]}")
if sobrando_no_humano:
    print(f"  sobram no humano (até 10): {sorted(sobrando_no_humano)[:10]}")

# 3) Conteúdo: para cada (pasta, id) casada, pergunta/resposta/categoria batem?
check = divergentes_chats.merge(
    df_human_chats[["pasta", "id", "categoria", "pergunta", "resposta"]],
    on=["pasta", "id"],
    how="inner",
    suffixes=("_result", "_human"),
)
for col in ["categoria", "pergunta", "resposta"]:
    igual = (check[f"{col}_result"] == check[f"{col}_human"]).all()
    print(f"`{col}` igual em todas as linhas casadas?           {igual}")
    if not igual:
        diff = check[check[f"{col}_result"] != check[f"{col}_human"]]
        print(f"  {len(diff)} divergência(s). Primeiras:")
        print(diff[["pasta", "id", f"{col}_result", f"{col}_human"]].head().to_string(index=False))

# 4) Duplicatas em (pasta, id) no humano
dup_human = df_human_chats.duplicated(subset=["pasta", "id"], keep=False)
if dup_human.any():
    print(f"\n[!] {dup_human.sum()} linha(s) com (pasta, id) duplicada no humano:")
    print(df_human_chats.loc[dup_human, ["pasta", "id"]].sort_values(["pasta", "id"]).to_string(index=False))


Colunas no arquivo humano (chats): ['edital', 'modelo', 'provedor', 'id', 'categoria', 'pergunta', 'resposta', 'avaliacao_gpt', 'justificativa_gpt', 'avaliacao_opus', 'justificativa_opus', 'convergencia', 'avaliação humana', 'modelo.1']

Divergentes em result_chats (convergencia=False): 29
Linhas no arquivo humano:                          29
Quantidade igual?                                  True
Chaves (pasta, id) idênticas?                      True
`categoria` igual em todas as linhas casadas?           True
`pergunta` igual em todas as linhas casadas?           True
`resposta` igual em todas as linhas casadas?           True


In [7]:
# --- Etapa 5.2: merge da avaliação humana em `result_chats` ---

# Renomeia para snake_case (consistência com avaliacao_gpt / avaliacao_opus)
df_human_chats_cols = df_human_chats.rename(columns={
    "avaliação humana": "avaliacao_humana",
    "modelo.1":         "modelo_correto",
})[["pasta", "id", "avaliacao_humana", "modelo_correto"]]

n_antes = len(result_chats)
result_chats = result_chats.merge(df_human_chats_cols, on=["pasta", "id"], how="left")

# Sanity checks ANTES do fillna (após o fillna, notna() retornaria sempre len(result_chats))
n_humanas = result_chats["avaliacao_humana"].notna().sum()
n_div     = (~result_chats["convergencia"]).sum()
print(f"Linhas em result_chats: {len(result_chats)} (antes: {n_antes})")
print(f"avaliacao_humana preenchida em: {n_humanas}")
print(f"Divergentes (convergencia=False): {n_div}")
print(f"Bate?                             {len(result_chats) == n_antes and n_humanas == n_div}")

# Preenche convergentes (sem divergência → sem avaliação humana) com placeholder
result_chats = result_chats.drop(columns=["pasta"]).fillna({
    "avaliacao_humana": "nao pertinente",
    "modelo_correto":   "nao pertinente",
})

result_chats.head()


Linhas em result_chats: 300 (antes: 300)
avaliacao_humana preenchida em: 29
Divergentes (convergencia=False): 29
Bate?                             True


,edital,modelo,id,categoria,pergunta,resposta,resposta_tokens_tiktoken,avaliacao_gpt,justificativa_gpt,avaliacao_opus,justificativa_opus,convergencia,avaliacao_humana,modelo_correto
0,bndes,std_chatgpt,1,Dados sobre as inscrições,Qual o período de inscrição para o concurso BN...,O período de inscrição para a Seleção Pública ...,98,1.0,Certo. O item 6.2.1 (p. 9) prevê inscrição ent...,1.0,Subitens 6.2.1 e 6.3.1 do edital: inscrições n...,True,nao pertinente,nao pertinente
1,bndes,std_chatgpt,2,Dados sobre as inscrições,Qual o valor da taxa de inscrição do concurso ...,O valor da taxa de inscrição do concurso do\nB...,40,1.0,Certo. O item 6.2.3 (p. 9) fixa o recolhimento...,1.0,Subitem 6.2.3 do edital: 'O recolhimento do va...,True,nao pertinente,nao pertinente
2,bndes,std_chatgpt,3,Dados sobre as inscrições,Em qual site o candidato deve fazer a inscriçã...,A inscrição do concurso do\nBanco Nacional de ...,46,1.0,Certo. Os itens 6.2.1 e 6.3.1 (p. 9-10) indica...,1.0,Subitem 6.2.1 do edital: inscrição na página d...,True,nao pertinente,nao pertinente
3,bndes,std_chatgpt,4,Dados sobre as inscrições,Quais são as formas de pagamento da taxa de in...,De acordo com o edital do concurso do\nBanco N...,127,1.0,Certo. O item 6.3.2 (p. 10) prevê duas formas ...,1.0,Subitem 6.3.2 do edital: as duas formas de pag...,True,nao pertinente,nao pertinente
4,bndes,std_chatgpt,5,Dados sobre as inscrições,Qual é o prazo-limite de pagamento da inscriçã...,O edital informa que o pagamento da inscrição ...,53,1.0,Certo. O item 6.3.2 (p. 10) determina pagament...,1.0,Subitem 6.3.2 do edital: pagamento deve ser fe...,True,nao pertinente,nao pertinente


## Etapa 6 — Geração dos batches para avaliação de concisão

Gera arquivos em formato **largo**, com uma linha por pergunta e uma coluna de resposta por modelo.

**Saída:** `eval/concisao/batches2eval/`

**Padrão dos arquivos:** `{edital}_batch_01.xlsx`, `{edital}_batch_02.xlsx`, ..., `{edital}_batch_05.xlsx`.

Exemplo: `bndes_batch_01.xlsx`.

**Formato dos arquivos:**

- 5 batches por edital;
- 10 linhas por batch;
- aba padrão `Sheet1`;
- sem colunas de score/justificativa;
- colunas finais exatamente nesta ordem:
  `id`, `categoria`, `pergunta`, `resposta_claude-haiku-4-5`, `resposta_claude-opus-4-7`, `resposta_claude-sonnet-4-6`, `resposta_deepseek-v4-flash`, `resposta_deepseek-v4-pro`, `resposta_gpt-4o-mini`, `resposta_gpt-5.4`, `resposta_gpt-5.4-mini`, `resposta_gpt-5.5`, `resposta_chatgpt`, `resposta_claude_chat`.

A célula abaixo parte dos objetos já consolidados nas etapas anteriores:

- `result`: respostas da ferramenta;
- `result_chats`: respostas dos chats comerciais.


In [8]:
# --- Etapa 6: geração dos batches para avaliação de concisão ---

BATCHES_OUT_DIR = EVAL_ROOT / "concisao" / "batches2eval"
BATCHES_OUT_DIR.mkdir(parents=True, exist_ok=True)

LINHAS_POR_BATCH = 10
BATCHES_POR_EDITAL = 5
ESPERADO_POR_EDITAL = LINHAS_POR_BATCH * BATCHES_POR_EDITAL

# Segurança: se False, não sobrescreve arquivos já existentes.
# Como o padrão final é {edital}_batch_01.xlsx, isto evita apagar arquivos já validados.
SOBRESCREVER_BATCHES = False

MODELOS_FERRAMENTA = [
    "claude-haiku-4-5",
    "claude-opus-4-7",
    "claude-sonnet-4-6",
    "deepseek-v4-flash",
    "deepseek-v4-pro",
    "gpt-4o-mini",
    "gpt-5.4",
    "gpt-5.4-mini",
    "gpt-5.5",
]

MODELOS_CHATS = ["chatgpt", "claude_chat"]
MODELOS_SAIDA = MODELOS_FERRAMENTA + MODELOS_CHATS

COLUNAS_SAIDA = ["id", "categoria", "pergunta"] + [f"resposta_{m}" for m in MODELOS_SAIDA]

CHAT_MODEL_MAP = {
    "std_chatgpt": "chatgpt",
    "chatgpt": "chatgpt",
    "gpt_chat": "chatgpt",
    "std_claude": "claude_chat",
    "claude": "claude_chat",
    "claude_chat": "claude_chat",
}


def _pivot_respostas(df, modelo_col="modelo", resposta_col="resposta", prefix="resposta_"):
    """Transforma respostas longas em formato largo por (edital, id)."""
    necessario = {"edital", "id", modelo_col, resposta_col}
    faltantes = necessario - set(df.columns)
    if faltantes:
        raise ValueError(f"Colunas ausentes para pivot: {sorted(faltantes)}")

    tmp = df[["edital", "id", modelo_col, resposta_col]].copy()

    dup = tmp.duplicated(subset=["edital", "id", modelo_col], keep=False)
    if dup.any():
        exemplos = tmp.loc[dup, ["edital", "id", modelo_col]].head(20)
        raise ValueError(
            "Há respostas duplicadas para o mesmo (edital, id, modelo). "
            "Exemplos:\n" + exemplos.to_string(index=False)
        )

    wide = (
        tmp.pivot(index=["edital", "id"], columns=modelo_col, values=resposta_col)
           .reset_index()
    )
    wide.columns.name = None

    rename = {
        c: f"{prefix}{c}"
        for c in wide.columns
        if c not in {"edital", "id"}
    }
    return wide.rename(columns=rename)


# Base canônica: uma linha por pergunta/edital, usando ferramenta e chats como fontes.
base_tool = result[["edital", "id", "categoria", "pergunta"]].drop_duplicates(["edital", "id"])
base_chat = result_chats[["edital", "id", "categoria", "pergunta"]].drop_duplicates(["edital", "id"])

base = base_tool.merge(base_chat, on=["edital", "id"], how="outer", suffixes=("_tool", "_chat"))
base["categoria"] = base["categoria_tool"].combine_first(base["categoria_chat"])
base["pergunta"] = base["pergunta_tool"].combine_first(base["pergunta_chat"])
base = base[["edital", "id", "categoria", "pergunta"]]

# Respostas da ferramenta.
respostas_ferramenta = _pivot_respostas(
    result[result["modelo"].isin(MODELOS_FERRAMENTA)].copy(),
    modelo_col="modelo",
    resposta_col="resposta",
)

# Respostas dos chats comerciais, normalizando nomes para as colunas finais pedidas.
chats_norm = result_chats.copy()
chats_norm["modelo_batch"] = chats_norm["modelo"].map(CHAT_MODEL_MAP).fillna(chats_norm["modelo"])
respostas_chats = _pivot_respostas(
    chats_norm[chats_norm["modelo_batch"].isin(MODELOS_CHATS)].copy(),
    modelo_col="modelo_batch",
    resposta_col="resposta",
)

# Junta tudo no formato largo.
batches_base = (
    base.merge(respostas_ferramenta, on=["edital", "id"], how="left")
        .merge(respostas_chats, on=["edital", "id"], how="left")
)

# Garante exatamente as colunas pedidas, mesmo que algum modelo ainda esteja ausente.
for col in COLUNAS_SAIDA:
    if col not in batches_base.columns:
        batches_base[col] = pd.NA

# Verificações de integridade antes de salvar.
dup_base = batches_base.duplicated(subset=["edital", "id"], keep=False)
if dup_base.any():
    exemplos = batches_base.loc[dup_base, ["edital", "id", "pergunta"]].head(20)
    raise ValueError(
        "Há duplicatas em (edital, id) depois da consolidação. "
        "Exemplos:\n" + exemplos.to_string(index=False)
    )

resposta_cols = [c for c in COLUNAS_SAIDA if c.startswith("resposta_")]
faltantes_por_coluna = batches_base[resposta_cols].isna().sum().sort_values(ascending=False)
if (faltantes_por_coluna > 0).any():
    print("[AVISO] Respostas faltantes por coluna:")
    print(faltantes_por_coluna[faltantes_por_coluna > 0].to_string())
else:
    print("Todas as colunas de resposta estão completas.")

arquivos_gerados = []
arquivos_pulados = []

for edital, df_edital in batches_base.groupby("edital", sort=True):
    df_edital = df_edital.sort_values("id").reset_index(drop=True)

    if len(df_edital) != ESPERADO_POR_EDITAL:
        raise ValueError(
            f"{edital}: esperado {ESPERADO_POR_EDITAL} perguntas "
            f"({BATCHES_POR_EDITAL} batches x {LINHAS_POR_BATCH}), "
            f"mas encontrei {len(df_edital)}."
        )

    for batch_idx in range(BATCHES_POR_EDITAL):
        inicio = batch_idx * LINHAS_POR_BATCH
        fim = inicio + LINHAS_POR_BATCH
        df_batch = df_edital.iloc[inicio:fim][COLUNAS_SAIDA].copy()

        if len(df_batch) != LINHAS_POR_BATCH:
            raise ValueError(
                f"{edital} batch {batch_idx + 1}: esperado {LINHAS_POR_BATCH} linhas, "
                f"mas encontrei {len(df_batch)}."
            )

        # Padrão final do batch: exemplo bndes_batch_01.xlsx
        out_path = BATCHES_OUT_DIR / f"{edital}_batch_{batch_idx + 1:02d}.xlsx"

        if out_path.exists() and not SOBRESCREVER_BATCHES:
            arquivos_pulados.append(out_path)
            print(f"[SKIP] {out_path.name} já existe")
            continue

        # Mantém o padrão do arquivo exemplo: uma aba padrão Sheet1 e somente as 14 colunas finais.
        df_batch.to_excel(out_path, index=False, sheet_name="Sheet1")
        arquivos_gerados.append(out_path)
        print(f"[OK] {out_path.name} ({len(df_batch)} linhas)")

print(f"\nTotal de arquivos gerados: {len(arquivos_gerados)}")
print(f"Total de arquivos pulados: {len(arquivos_pulados)}")
print(f"Pasta de saída: {BATCHES_OUT_DIR}")


Todas as colunas de resposta estão completas.
[SKIP] bndes_batch_01.xlsx já existe
[SKIP] bndes_batch_02.xlsx já existe
[SKIP] bndes_batch_03.xlsx já existe
[SKIP] bndes_batch_04.xlsx já existe
[SKIP] bndes_batch_05.xlsx já existe
[SKIP] cvm_batch_01.xlsx já existe
[SKIP] cvm_batch_02.xlsx já existe
[SKIP] cvm_batch_03.xlsx já existe
[SKIP] cvm_batch_04.xlsx já existe
[SKIP] cvm_batch_05.xlsx já existe
[SKIP] petrobras_batch_01.xlsx já existe
[SKIP] petrobras_batch_02.xlsx já existe
[SKIP] petrobras_batch_03.xlsx já existe
[SKIP] petrobras_batch_04.xlsx já existe
[SKIP] petrobras_batch_05.xlsx já existe

Total de arquivos gerados: 0
Total de arquivos pulados: 15
Pasta de saída: /home/julio/Documentos/tcc_GENAI/v8/edital-assistant/eval/concisao/batches2eval


## Etapa 7 — Incorporação da avaliação de concisão e geração dos artefatos finais

Esta etapa lê os arquivos já avaliados em:

```text
/home/julio/Documentos/tcc_GENAI/v8/edital-assistant/eval/concisao/batches_avaliados_concisao
```

Formato esperado dos arquivos avaliados, como no exemplo `bndes_batch_01_avaliacao_foco_concisao.xlsx`:

```text
id | pergunta | modelo | resposta | score | justificativa
```

A partir disso, o notebook adiciona aos dataframes finais:

```text
avaliacao_final
concisao_score
justificativa_concisao
```

A coluna `avaliacao_final` fica posicionada entre `modelo_correto` e `concisao_score`.

Regra de `avaliacao_final`:

- se `avaliacao_gpt == avaliacao_opus`, usa a avaliação convergente;
- se há divergência, usa `avaliacao_humana`.

Os artefatos mantêm as colunas de token/custo em ordem legível:

```text
resposta_tokens_tiktoken | input_tokens | output_tokens | custo_estimado_usd
```

Por fim, salva em:

```text
/home/julio/Documentos/tcc_GENAI/v8/edital-assistant/eval/artefatos
```

os três arquivos:

```text
result_ferramenta_final.xlsx
result_chats_final.xlsx
result_unificado_final.xlsx
```

No `result_unificado_final.xlsx`, colunas ausentes são preenchidas com `nao pertinente`.


In [9]:
# --- Etapa 7: concisão, avaliação final e artefatos finais ---

CONCISAO_AVALIADOS_DIR = EVAL_ROOT / "concisao" / "batches_avaliados_concisao"
ARTEFATOS_DIR = EVAL_ROOT / "artefatos"
ARTEFATOS_DIR.mkdir(parents=True, exist_ok=True)

NAO_PERTINENTE = "nao pertinente"
SALVAR_ARTEFATOS = True

# Aceita tanto o padrão avaliado quanto o padrão sem sufixo.
CONCISAO_FILE_PATTERN = re.compile(
    r"^(?P<edital>[^_]+)_batch_(?P<batch>\d{2})(?:_avaliacao_foco_concisao)?\.xlsx$"
)

COLS_CONCISAO_ESPERADAS = {"id", "pergunta", "modelo", "resposta", "score", "justificativa"}

# Recria os mapas caso esta etapa seja executada isoladamente depois da Etapa 5.
if "CHAT_MODEL_MAP" not in globals():
    CHAT_MODEL_MAP = {
        "std_chatgpt": "chatgpt",
        "chatgpt": "chatgpt",
        "gpt_chat": "chatgpt",
        "std_claude": "claude_chat",
        "claude": "claude_chat",
        "claude_chat": "claude_chat",
    }

if "MODELOS_FERRAMENTA" not in globals():
    MODELOS_FERRAMENTA = [
        "claude-haiku-4-5",
        "claude-opus-4-7",
        "claude-sonnet-4-6",
        "deepseek-v4-flash",
        "deepseek-v4-pro",
        "gpt-4o-mini",
        "gpt-5.4",
        "gpt-5.4-mini",
        "gpt-5.5",
    ]

if "MODELOS_CHATS" not in globals():
    MODELOS_CHATS = ["chatgpt", "claude_chat"]


def _limpar_colunas_excel(df):
    """Remove colunas vazias típicas do Excel e normaliza nomes."""
    df = df.drop(columns=[c for c in df.columns if str(c).startswith("Unnamed:")], errors="ignore")
    df = df.rename(columns={c: str(c).strip() for c in df.columns})
    return df


def _ler_avaliacoes_concisao(pasta):
    if not pasta.exists():
        raise FileNotFoundError(f"Pasta não encontrada: {pasta}")

    frames = []
    arquivos_lidos = []
    arquivos_ignorados = []

    for path in sorted(pasta.glob("*.xlsx")):
        m = CONCISAO_FILE_PATTERN.match(path.name)
        if not m:
            arquivos_ignorados.append(path.name)
            print(f"[SKIP] {path.name}: nome fora do padrão esperado")
            continue

        df = _limpar_colunas_excel(pd.read_excel(path))
        faltantes = COLS_CONCISAO_ESPERADAS - set(df.columns)
        if faltantes:
            raise ValueError(
                f"{path.name}: colunas ausentes {sorted(faltantes)}. "
                f"Colunas encontradas: {list(df.columns)}"
            )

        df = df[["id", "pergunta", "modelo", "resposta", "score", "justificativa"]].copy()
        df.insert(0, "edital", m.group("edital"))
        df.insert(1, "batch", int(m.group("batch")))
        df["arquivo_origem"] = path.name
        frames.append(df)
        arquivos_lidos.append(path.name)

    if not frames:
        raise ValueError(f"Nenhum arquivo de concisão válido encontrado em: {pasta}")

    concisao = pd.concat(frames, ignore_index=True)

    # Normalizações leves para evitar falhas por tipo/espaço.
    concisao["id"] = pd.to_numeric(concisao["id"], errors="raise").astype(int)
    concisao["modelo"] = concisao["modelo"].astype(str).str.strip()
    concisao["score"] = pd.to_numeric(concisao["score"], errors="coerce").astype("Int64")
    concisao["justificativa"] = concisao["justificativa"].astype("string")

    # Renomeia para o padrão final que vai entrar nos resultados.
    concisao = concisao.rename(columns={
        "score": "concisao_score",
        "justificativa": "justificativa_concisao",
    })

    # Uma avaliação por resposta: edital + id + modelo.
    dup = concisao.duplicated(subset=["edital", "id", "modelo"], keep=False)
    if dup.any():
        exemplos = concisao.loc[dup, ["edital", "batch", "id", "modelo", "arquivo_origem"]]\
                           .sort_values(["edital", "id", "modelo", "batch"])\
                           .head(30)
        raise ValueError(
            "Há duplicatas de avaliação de concisão para o mesmo (edital, id, modelo).\n"
            + exemplos.to_string(index=False)
        )

    print(f"Arquivos de concisão lidos: {len(arquivos_lidos)}")
    if arquivos_ignorados:
        print(f"Arquivos ignorados: {len(arquivos_ignorados)}")

    print("Linhas de avaliação de concisão:", len(concisao))
    print("Resumo por edital:")
    print(concisao.groupby("edital").size().to_string())

    print("\nResumo por modelo:")
    print(concisao.groupby("modelo").size().sort_index().to_string())

    return concisao


concisao = _ler_avaliacoes_concisao(CONCISAO_AVALIADOS_DIR)

# Dataset mínimo para merge, evitando levar pergunta/resposta duplicadas para os resultados.
concisao_merge = concisao[[
    "edital", "id", "modelo", "concisao_score", "justificativa_concisao"
]].copy()


def _anexar_concisao(df, nome_df, modelo_map=None):
    """Adiciona concisao_score e justificativa_concisao a um resultado consolidado."""
    if df is None or len(df) == 0:
        raise ValueError(f"{nome_df} está vazio ou não foi criado.")

    cols_obrigatorias = {"edital", "id", "modelo"}
    faltantes = cols_obrigatorias - set(df.columns)
    if faltantes:
        raise ValueError(f"{nome_df}: colunas ausentes para merge: {sorted(faltantes)}")

    out = df.copy()

    # Reexecução segura da célula: remove colunas antigas antes de recalcular.
    out = out.drop(columns=["concisao_score", "justificativa_concisao"], errors="ignore")

    out["id"] = pd.to_numeric(out["id"], errors="raise").astype(int)
    out["_modelo_concisao"] = out["modelo"].astype(str).str.strip()
    if modelo_map:
        out["_modelo_concisao"] = out["_modelo_concisao"].map(modelo_map).fillna(out["_modelo_concisao"])

    before = len(out)
    merged = out.merge(
        concisao_merge.rename(columns={"modelo": "_modelo_concisao"}),
        on=["edital", "id", "_modelo_concisao"],
        how="left",
        validate="many_to_one",
    )

    if len(merged) != before:
        raise ValueError(f"{nome_df}: o merge alterou a quantidade de linhas ({before} -> {len(merged)}).")

    n_sem = merged["concisao_score"].isna().sum()
    print(f"{nome_df}: {len(merged) - n_sem}/{len(merged)} linhas com concisao_score.")

    if n_sem:
        print(f"[AVISO] {nome_df}: {n_sem} linha(s) sem avaliação de concisão. Primeiras chaves:")
        cols_exemplo = ["edital", "id", "modelo", "_modelo_concisao"]
        print(merged.loc[merged["concisao_score"].isna(), cols_exemplo].head(20).to_string(index=False))

    return merged.drop(columns=["_modelo_concisao"])


def _adicionar_avaliacao_final(df, nome_df):
    """
    Cria avaliacao_final:
    - convergentes: usa avaliacao_gpt/avaliacao_opus, que são iguais;
    - divergentes: usa avaliacao_humana.
    """
    obrigatorias = {"avaliacao_gpt", "avaliacao_opus", "convergencia", "avaliacao_humana"}
    faltantes = obrigatorias - set(df.columns)
    if faltantes:
        raise ValueError(f"{nome_df}: colunas ausentes para avaliacao_final: {sorted(faltantes)}")

    out = df.drop(columns=["avaliacao_final"], errors="ignore").copy()

    conv = out["convergencia"] == True
    div = out["convergencia"] == False

    # Checagem defensiva: se converge, GPT e Opus deveriam ser iguais.
    conv_desalinhada = conv & (out["avaliacao_gpt"] != out["avaliacao_opus"])
    if conv_desalinhada.any():
        print(
            f"[AVISO] {nome_df}: {conv_desalinhada.sum()} linha(s) com convergencia=True, "
            "mas avaliacao_gpt != avaliacao_opus. A avaliacao_final usará avaliacao_gpt."
        )

    # Checagem defensiva: se diverge, precisa haver avaliação humana real.
    humana_txt = out["avaliacao_humana"].astype("string").str.strip().str.lower()
    div_sem_humana = div & (out["avaliacao_humana"].isna() | humana_txt.isin(["", NAO_PERTINENTE]))
    if div_sem_humana.any():
        print(
            f"[AVISO] {nome_df}: {div_sem_humana.sum()} linha(s) divergente(s) sem avaliacao_humana real. "
            f"A avaliacao_final ficará como {NAO_PERTINENTE} nessas linhas."
        )

    out["avaliacao_final"] = pd.NA
    out.loc[conv, "avaliacao_final"] = out.loc[conv, "avaliacao_gpt"]
    out.loc[div, "avaliacao_final"] = out.loc[div, "avaliacao_humana"]
    out["avaliacao_final"] = out["avaliacao_final"].fillna(NAO_PERTINENTE)

    return _reordenar_bloco_avaliacao(out)


def _reordenar_bloco_avaliacao(df):
    """Mantém modelo_correto, avaliacao_final, concisao_score e justificativa_concisao juntos."""
    bloco = [
        c for c in ["avaliacao_final", "concisao_score", "justificativa_concisao"]
        if c in df.columns
    ]
    if not bloco or "modelo_correto" not in df.columns:
        return df

    cols = [c for c in df.columns if c not in bloco]
    pos = cols.index("modelo_correto") + 1
    cols = cols[:pos] + bloco + cols[pos:]
    return df[cols]


def _reordenar_colunas_artefato(df):
    """Ordem final estável para exportação dos artefatos."""
    preferidas = [
        "origem_resultado",
        "edital", "modelo", "id", "categoria", "pergunta", "resposta",
        "resposta_tokens_tiktoken",
        "input_tokens", "output_tokens", "custo_estimado_usd",
        "n_invocacoes", "latencia_s",
        "avaliacao_gpt", "justificativa_gpt", "avaliacao_opus", "justificativa_opus",
        "convergencia", "avaliacao_humana", "modelo_correto", "avaliacao_final",
        "concisao_score", "justificativa_concisao",
    ]
    cols = [c for c in preferidas if c in df.columns]
    cols += [c for c in df.columns if c not in cols]
    return df[cols]


# Resultados da ferramenta: modelos já estão com os mesmos nomes usados nos batches.
result = _anexar_concisao(result, "result")
result = _adicionar_avaliacao_final(result, "result")

# Resultados dos chats: normaliza nomes para chatgpt/claude_chat antes do merge.
result_chats = _anexar_concisao(result_chats, "result_chats", modelo_map=CHAT_MODEL_MAP)
result_chats = _adicionar_avaliacao_final(result_chats, "result_chats")

# Artefatos individuais.
result = _reordenar_colunas_artefato(result)
result_chats = _reordenar_colunas_artefato(result_chats)

result_ferramenta_art = result.copy()
result_chats_art = result_chats.copy()

# Artefato unificado. O marcador de origem é útil porque os dois dfs passam a conviver no mesmo arquivo.
result_ferramenta_unif = result.copy()
result_ferramenta_unif.insert(0, "origem_resultado", "ferramenta")

result_chats_unif = result_chats.copy()
result_chats_unif.insert(0, "origem_resultado", "chat_comercial")

result_unificado = pd.concat(
    [result_ferramenta_unif, result_chats_unif],
    ignore_index=True,
    sort=False,
)

# Como o dataframe unificado mistura colunas numéricas da ferramenta com colunas ausentes nos chats,
# convertemos para object antes de preencher com texto. Isso evita erro em colunas Int64/Float64.
result_unificado = result_unificado.astype("object").fillna(NAO_PERTINENTE)

# Reordena também no unificado, caso o concat tenha deslocado alguma coluna.
result_unificado = _reordenar_bloco_avaliacao(result_unificado)
result_unificado = _reordenar_colunas_artefato(result_unificado)

if SALVAR_ARTEFATOS:
    ferramenta_out = ARTEFATOS_DIR / "result_ferramenta_final.xlsx"
    chats_out = ARTEFATOS_DIR / "result_chats_final.xlsx"
    unificado_out = ARTEFATOS_DIR / "result_unificado_final.xlsx"

    result_ferramenta_art.to_excel(ferramenta_out, index=False)
    result_chats_art.to_excel(chats_out, index=False)
    result_unificado.to_excel(unificado_out, index=False)

    print("\nArtefatos salvos:")
    print(ferramenta_out)
    print(chats_out)
    print(unificado_out)

print("\nShapes finais:")
print(f"result:           {result.shape}")
print(f"result_chats:     {result_chats.shape}")
print(f"result_unificado: {result_unificado.shape}")

print("\nPreview result:")
display(result.head(3))

print("\nPreview result_chats:")
display(result_chats.head(3))

print("\nPreview result_unificado:")
display(result_unificado.head(3))


Arquivos de concisão lidos: 15
Linhas de avaliação de concisão: 1650
Resumo por edital:
edital
bndes        550
cvm          550
petrobras    550

Resumo por modelo:
modelo
chatgpt              150
claude-haiku-4-5     150
claude-opus-4-7      150
claude-sonnet-4-6    150
claude_chat          150
deepseek-v4-flash    150
deepseek-v4-pro      150
gpt-4o-mini          150
gpt-5.4              150
gpt-5.4-mini         150
gpt-5.5              150
result: 1350/1350 linhas com concisao_score.
result_chats: 300/300 linhas com concisao_score.

Artefatos salvos:
/home/julio/Documentos/tcc_GENAI/v8/edital-assistant/eval/artefatos/result_ferramenta_final.xlsx
/home/julio/Documentos/tcc_GENAI/v8/edital-assistant/eval/artefatos/result_chats_final.xlsx
/home/julio/Documentos/tcc_GENAI/v8/edital-assistant/eval/artefatos/result_unificado_final.xlsx

Shapes finais:
result:           (1350, 22)
result_chats:     (300, 17)
result_unificado: (1650, 23)

Preview result:


,edital,modelo,id,categoria,pergunta,resposta,resposta_tokens_tiktoken,input_tokens,output_tokens,custo_estimado_usd,...,avaliacao_gpt,justificativa_gpt,avaliacao_opus,justificativa_opus,convergencia,avaliacao_humana,modelo_correto,avaliacao_final,concisao_score,justificativa_concisao
0,bndes,claude-haiku-4-5,1,Dados sobre as inscrições,Qual o período de inscrição para o concurso BN...,**Período de inscrição do BNDES (01/2024):**\n...,143,10951,307,0.012486,...,1.0,A resposta está alinhada ao edital: o item 6.2...,1.0,"O edital, no item 6.2.1, estabelece o período ...",True,nao pertinente,nao pertinente,1.0,1,respondeu o período com aviso útil e volume ra...
1,bndes,claude-haiku-4-5,2,Dados sobre as inscrições,Qual o valor da taxa de inscrição do concurso ...,A **taxa de inscrição do concurso BNDES é de R...,203,10440,362,0.01225,...,1.0,Correta. O item 6.2.3 fixa o valor de inscriçã...,1.0,O item 6.2.3 fixa expressamente o valor de R$ ...,True,nao pertinente,nao pertinente,1.0,0,"despejou pagamento, devolução e isenção sem pedir"
2,bndes,claude-haiku-4-5,3,Dados sobre as inscrições,Em qual site o candidato deve fazer a inscriçã...,O candidato deve fazer a inscrição no **site d...,113,10079,257,0.011364,...,1.0,Correta. Os itens 6.2.1 e 6.3.1 determinam que...,1.0,O item 6.2.1 e o 6.3.1 indicam expressamente q...,True,nao pertinente,nao pertinente,1.0,0,"despejou período, escolha de cargo e taxa sem ..."



Preview result_chats:


,edital,modelo,id,categoria,pergunta,resposta,resposta_tokens_tiktoken,avaliacao_gpt,justificativa_gpt,avaliacao_opus,justificativa_opus,convergencia,avaliacao_humana,modelo_correto,avaliacao_final,concisao_score,justificativa_concisao
0,bndes,std_chatgpt,1,Dados sobre as inscrições,Qual o período de inscrição para o concurso BN...,O período de inscrição para a Seleção Pública ...,98,1.0,Certo. O item 6.2.1 (p. 9) prevê inscrição ent...,1.0,Subitens 6.2.1 e 6.3.1 do edital: inscrições n...,True,nao pertinente,nao pertinente,1.0,1,respondeu o período com volume razoável
1,bndes,std_chatgpt,2,Dados sobre as inscrições,Qual o valor da taxa de inscrição do concurso ...,O valor da taxa de inscrição do concurso do\nB...,40,1.0,Certo. O item 6.2.3 (p. 9) fixa o recolhimento...,1.0,Subitem 6.2.3 do edital: 'O recolhimento do va...,True,nao pertinente,nao pertinente,1.0,1,respondeu só a taxa com volume razoável
2,bndes,std_chatgpt,3,Dados sobre as inscrições,Em qual site o candidato deve fazer a inscriçã...,A inscrição do concurso do\nBanco Nacional de ...,46,1.0,Certo. Os itens 6.2.1 e 6.3.1 (p. 9-10) indica...,1.0,Subitem 6.2.1 do edital: inscrição na página d...,True,nao pertinente,nao pertinente,1.0,1,respondeu o site com volume razoável



Preview result_unificado:


,origem_resultado,edital,modelo,id,categoria,pergunta,resposta,resposta_tokens_tiktoken,input_tokens,output_tokens,...,avaliacao_gpt,justificativa_gpt,avaliacao_opus,justificativa_opus,convergencia,avaliacao_humana,modelo_correto,avaliacao_final,concisao_score,justificativa_concisao
0,ferramenta,bndes,claude-haiku-4-5,1,Dados sobre as inscrições,Qual o período de inscrição para o concurso BN...,**Período de inscrição do BNDES (01/2024):**\n...,143,10951,307,...,1.0,A resposta está alinhada ao edital: o item 6.2...,1.0,"O edital, no item 6.2.1, estabelece o período ...",True,nao pertinente,nao pertinente,1.0,1,respondeu o período com aviso útil e volume ra...
1,ferramenta,bndes,claude-haiku-4-5,2,Dados sobre as inscrições,Qual o valor da taxa de inscrição do concurso ...,A **taxa de inscrição do concurso BNDES é de R...,203,10440,362,...,1.0,Correta. O item 6.2.3 fixa o valor de inscriçã...,1.0,O item 6.2.3 fixa expressamente o valor de R$ ...,True,nao pertinente,nao pertinente,1.0,0,"despejou pagamento, devolução e isenção sem pedir"
2,ferramenta,bndes,claude-haiku-4-5,3,Dados sobre as inscrições,Em qual site o candidato deve fazer a inscriçã...,O candidato deve fazer a inscrição no **site d...,113,10079,257,...,1.0,Correta. Os itens 6.2.1 e 6.3.1 determinam que...,1.0,O item 6.2.1 e o 6.3.1 indicam expressamente q...,True,nao pertinente,nao pertinente,1.0,0,"despejou período, escolha de cargo e taxa sem ..."


In [10]:
result_unificado[result_unificado['modelo'].isin(['std_chatgpt', 'std_claude'])].tail(3)

,origem_resultado,edital,modelo,id,categoria,pergunta,resposta,resposta_tokens_tiktoken,input_tokens,output_tokens,...,avaliacao_gpt,justificativa_gpt,avaliacao_opus,justificativa_opus,convergencia,avaliacao_humana,modelo_correto,avaliacao_final,concisao_score,justificativa_concisao
1647,chat_comercial,petrobras,std_claude,48,Dados sobre Procedimentos,Qual tipo de caneta pode ser utilizado para re...,"De acordo com o\nitem 8.1\ndo edital, o tipo d...",511,nao pertinente,nao pertinente,...,1.0,Correta. O item 8.1 (p. 13) permite apenas can...,1.0,Item 8.1 confirma: 'munido(a) somente de canet...,True,nao pertinente,nao pertinente,1.0,0,despejou folha de respostas e consequências se...
1648,chat_comercial,petrobras,std_claude,49,Dados sobre Procedimentos,Como o candidato deve se identificar no dia da...,"De acordo com o edital, veja todas as informaç...",1017,nao pertinente,nao pertinente,...,1.0,Correta. O item 8.1 (p. 13) exige documento de...,1.0,Item 8.2 confirma a lista de documentos aceito...,True,nao pertinente,nao pertinente,1.0,0,despejou detector de metal e nome social sem p...
1649,chat_comercial,petrobras,std_claude,50,Dados sobre Procedimentos,Quais cuidados o candidato deve ter com o cart...,"De acordo com o edital, veja todos os cuidados...",1148,nao pertinente,nao pertinente,...,1.0,Correta. Os itens 7.3 a 7.9 (p. 12) tratam da ...,1.0,Item 7.4 confirma a importância da folha de re...,True,nao pertinente,nao pertinente,1.0,0,"despejou consulta, pontuação e causas de elimi..."


In [11]:
result_unificado.columns

Index(['origem_resultado', 'edital', 'modelo', 'id', 'categoria', 'pergunta',
       'resposta', 'resposta_tokens_tiktoken', 'input_tokens', 'output_tokens',
       'custo_estimado_usd', 'n_invocacoes', 'latencia_s', 'avaliacao_gpt',
       'justificativa_gpt', 'avaliacao_opus', 'justificativa_opus',
       'convergencia', 'avaliacao_humana', 'modelo_correto', 'avaliacao_final',
       'concisao_score', 'justificativa_concisao'],
      dtype='str')